# 01 · Dual encoder from scratch

Build a CLIP-style dual encoder (text tower + image tower + shared projection) with plain PyTorch on the synthetic product corpus. This is the maths behind `app/embed.py` + `app/projection.py` in production.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('../..'))  # repo root
sys.path.insert(0, os.path.abspath('../../services/model-server'))
import torch, numpy as np
from ml.contrastive import info_nce, SharedSpaceTrainer

# CONFIG — scale up for real training
CONFIG = dict(epochs=3, batch=32, dim=64, lr=1e-3, temperature=0.07, seed=42)
torch.manual_seed(CONFIG['seed']); np.random.seed(CONFIG['seed'])
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## Data: hashed text + placeholder images as raw features

We skip pretrained encoders here on purpose — the point is the **contrastive shared-space objective**, not the towers.

In [ ]:
from PIL import Image
import csv

manifest = '../../ml/data/generated/manifest.csv'
rows = list(csv.DictReader(open(manifest)))[:512] if os.path.exists(manifest) else []
if not rows:
    raise SystemExit('run: python ml/data/make_synthetic_dataset.py first')

def text_features(t):
    v = np.zeros(CONFIG['dim'], np.float32)
    for tok in t.lower().split():
        v[hash(tok) % CONFIG['dim']] += 1.0
    return v / (np.linalg.norm(v) + 1e-8)

def image_features(path):
    a = np.asarray(Image.open(path).resize((16, 16))).astype(np.float32) / 255
    return a.reshape(-1)[:CONFIG['dim']] if a.size >= CONFIG['dim'] else np.pad(a.reshape(-1), (0, CONFIG['dim'] - a.size))

T = torch.tensor(np.stack([text_features(r['description']) for r in rows]), device=device)
I = torch.tensor(np.stack([image_features(os.path.join('../../ml/data/generated', r['image_path'])) for r in rows]), device=device)
print('pairs:', tuple(T.shape), tuple(I.shape))

## Train the shared space with symmetric InfoNCE

In [ ]:
proj = torch.nn.Linear(CONFIG['dim'], CONFIG['dim'], bias=False).to(device)
trainer = SharedSpaceTrainer(proj, lr=CONFIG['lr'])
history = []
for epoch in range(CONFIG['epochs']):
    perm = torch.randperm(len(T), device=device)
    for i in range(0, len(T) - CONFIG['batch'], CONFIG['batch']):
        idx = perm[i:i + CONFIG['batch']]
        loss, metrics = trainer.step(T[idx], I[idx], CONFIG['temperature'])
        history.append(loss)
print('final loss:', history[-1])

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history); plt.xlabel('step'); plt.ylabel('InfoNCE loss')
plt.title('shared-space training'); plt.show()

## Retrieval demo: text → image and image → text

In [ ]:
with torch.no_grad():
    t = torch.nn.functional.normalize(proj(T), dim=-1)
    i = torch.nn.functional.normalize(proj(I), dim=-1)
scores = t @ i.t()
print('image→text top1 acc:', (scores.argmax(1) == torch.arange(len(t), device=device)).float().mean().item())
print('text→image top1 acc:', (scores.argmax(0) == torch.arange(len(t), device=device)).float().mean().item())
print('sample ranks for item 0:', (scores[0] > scores[0, 0]).sum().item() + 1)